In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error

# ============================================================
# 1) Data read-in + validation
# ============================================================
df = pd.read_csv("claims_train.csv")
df.columns = [c.strip() for c in df.columns]

required = {
    'IDpol','ClaimNb','Exposure','VehBrand','VehGas','VehPower','VehAge', 'DrivAge','Area','Density','Region','BonusMalus'
}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing expected columns: {missing}")


# ============================================================
# 2) Target variables
# ============================================================
df['Exposure'] = df['Exposure'].clip(upper=1.0)
df['ClaimRate'] = df['ClaimNb'] / df['Exposure']

# ============================================================
# 3) Feature Engineering Transformer
# ============================================================
class FeatureRules(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.bonus_bins = [0, 100, 150, 200, np.inf]
        self.bonus_labels = ["bonus", "mild_malus", "malus", "heavy_malus"]
        self.driv_bins = [17, 25, 35, 50, 65, 120]
        self.driv_labels = ["18-25","26-35","36-50","51-65","65+"]
        self.vehage_bins = [-1, 2, 5, 10, np.inf]
        self.vehage_labels = ["0-2","3-5","6-10","10+"]
        self.area_order = ['A','B','C','D','E','F']

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        # Density: log(1+p)
        X['Density_log1p'] = np.log1p(X['Density'].astype(float))

        # BonusMalus: binned
        X['BonusMalus_bin'] = pd.cut(
            X['BonusMalus'], bins=self.bonus_bins,
            labels=self.bonus_labels, include_lowest=True
        )

        # Driver Age: binned
        X['DrivAge_bin'] = pd.cut(
            X['DrivAge'], bins=self.driv_bins,
            labels=self.driv_labels, include_lowest=True
        )

        # Vehicle Age: binned
        X['VehAge_bin'] = pd.cut(
            X['VehAge'], bins=self.vehage_bins,
            labels=self.vehage_labels, include_lowest=True
        )

        # VehPower: include both numeric rank and quantile bins
        X['VehPower_rank'] = X['VehPower'].rank(method='average')
        X['VehPower_bin'] = pd.qcut(
            X['VehPower'], q=4,
            labels=['low','mid','high','very_high']
        )

        # Area: ordered ordinal
        area_map = {a:i+1 for i,a in enumerate(self.area_order)}
        X['Area_ord'] = X['Area'].map(area_map).fillna(np.median(list(area_map.values())))

        # VehGas: binary
        X['VehGas_bin'] = (X['VehGas'] == 'Diesel').astype(int)

        return X

# Apply transformation
rules = FeatureRules()
df_feat = rules.transform(df)

# ============================================================
# 4) Preprocessing setup
# ============================================================
keep_num = ['Density_log1p', 'VehPower', 'VehPower_rank', 'Area_ord', 'VehGas_bin', 'Exposure']
ohe_cols = ['Region', 'VehBrand']
bin_cat_cols = ['BonusMalus_bin', 'DrivAge_bin', 'VehAge_bin', 'VehPower_bin']

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('sc', StandardScaler())
        ]), keep_num),
        ('ohe', Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('enc', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
        ]), ohe_cols + bin_cat_cols),
    ],
    remainder='drop'
)

# ============================================================
# 5) PCA: cumulative explained variance
# ============================================================
X_mat = preprocess.fit_transform(df_feat)
pca = PCA().fit(X_mat)

cum_var = np.cumsum(pca.explained_variance_ratio_)
plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, len(cum_var)+1), cum_var, marker='o')
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA - Cumulative Explained Variance")
plt.grid(True)
plt.show()

# Identify "important" original features via absolute loadings on PC1+PC2
loadings = np.abs(pca.components_[:2, :])  # first two PCs
importance = loadings.sum(axis=0)
top_idx = np.argsort(importance)[-20:][::-1]
top_features = [X_mat[i] for i in top_idx]
print("Top features by PCA loading on PC1+PC2:", top_features)

# ============================================================
# 6) Clustering (still possible)
# ============================================================

# USE 3D CLUSTERING

kmeans = KMeans(n_clusters=4, random_state=0, n_init=10)
labels = kmeans.fit_predict(X_mat[:, :3])  # top 3 components
plt.figure()
plt.scatter(X_mat[:,0], X_mat[:,1], c=labels, alpha=0.6)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("KMeans Clusters (First 2 PCA Components)")
plt.show()

# ============================================================
# 7) Modeling: Decision Tree & Neural Network
# ============================================================

# creata a parametric setup for models (for our 3rd)
# find a way to assess model performance (eg. having a lot of 0s gives a low error, but not useful)
# include the test set and cross-validation
# use parametric features instead of hard-coding
# find a way to print the process of model training

y_claimNb = df['ClaimNb']
y_claimRate = df['ClaimRate']

# --- Decision Tree Regressors ---
tree_nb = Pipeline([
    ('prep', preprocess),
    ('model', DecisionTreeRegressor(max_depth=8, random_state=0))
])
tree_nb.fit(df_feat, y_claimNb)
pred_nb = tree_nb.predict(df_feat)
print(f"[Decision Tree] ClaimNb MAE: {mean_absolute_error(y_claimNb, pred_nb):.4f}")

tree_rate = Pipeline([
    ('prep', preprocess),
    ('model', DecisionTreeRegressor(max_depth=8, random_state=0))
])
tree_rate.fit(df_feat, y_claimRate)
pred_rate = tree_rate.predict(df_feat)
print(f"[Decision Tree] ClaimRate MAE: {mean_absolute_error(y_claimRate, pred_rate):.6f}")

# --- Neural Network Regressors ---
nn_nb = Pipeline([
    ('prep', preprocess),
    ('model', MLPRegressor(hidden_layer_sizes=(64,32), max_iter=500, random_state=0))
])
nn_nb.fit(df_feat, y_claimNb)
pred_nn_nb = nn_nb.predict(df_feat)
print(f"[Neural Net] ClaimNb MAE: {mean_absolute_error(y_claimNb, pred_nn_nb):.4f}")

nn_rate = Pipeline([
    ('prep', preprocess),
    ('model', MLPRegressor(hidden_layer_sizes=(64,32), max_iter=500, random_state=0))
])
nn_rate.fit(df_feat, y_claimRate)
pred_nn_rate = nn_rate.predict(df_feat)
print(f"[Neural Net] ClaimRate MAE: {mean_absolute_error(y_claimRate, pred_nn_rate):.6f}")


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  # needed for 3D

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import mean_absolute_error, make_scorer

from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import GradientBoostingRegressor, ExtraTreesRegressor

# ============================================================
# 0) Minimal config (parametric schema, fewer hard-coded names)
# ============================================================
SCHEMA = {
    "id_col": "IDpol",
    "targets": {"nb": "ClaimNb", "rate": "ClaimRate"},
    "exposure": "Exposure",
    "raw_numeric_candidates": ["VehPower", "VehAge", "DrivAge", "Density"],  # will be filtered if missing
    "categorical_candidates": ["VehBrand", "VehGas", "Region", "Area", "VehGas_bin", 
                               "BonusMalus_bin", "DrivAge_bin", "VehAge_bin", "VehPower_bin"],
    "ordered_area": ['A','B','C','D','E','F'],
    "bonus_bins": ([0, 100, 150, 200, np.inf], ["bonus","mild_malus","malus","heavy_malus"]),
    "driv_bins": ([17, 25, 35, 50, 65, 120], ["18-25","26-35","36-50","51-65","65+"]),
    "vehage_bins": ([-1, 2, 5, 10, np.inf], ["0-2","3-5","6-10","10+"]),
    "vehpower_q": 4,                # quantile bins for VehPower_bin
    "pca_var_threshold": 0.95,      # use PCs up to this cumulative explained variance
    "alpha_nonzero_penalty": 4.0,   # weight multiplier for non-zero target in custom MAE
}

# ============================================================
# 1) Read data (train + optional test); basic checks
# ============================================================
train_path = "/Data/claims_train.csv" if os.path.exists("/Data/claims_train.csv") else "claims_train.csv"
test_path  = "/Data/claims_test.csv"  if os.path.exists("/Data/claims_test.csv")  else "claims_test.csv"

df_train = pd.read_csv(train_path)
df_train.columns = df_train.columns.str.strip()

has_external_test = os.path.exists(test_path)
df_test = pd.read_csv(test_path).pipe(lambda d: d.assign(**{c: d[c] for c in d.columns})) if has_external_test else None
if has_external_test:
    df_test.columns = df_test.columns.str.strip()

# Drop exposure == 0 (no time at risk)
df_train = df_train[df_train["Exposure"] > 0].copy()
if has_external_test:
    df_test = df_test[df_test["Exposure"] > 0].copy()

# Targets
df_train["Exposure"] = df_train["Exposure"].clip(upper=1.0)
if has_external_test:
    df_test["Exposure"] = df_test["Exposure"].clip(upper=1.0)

df_train["ClaimRate"] = df_train["ClaimNb"] / df_train["Exposure"]

if has_external_test:
    df_test["Exposure"] = df_test["Exposure"].clip(upper=1.0)
    # test won't have ClaimNb usually; only compute ClaimRate if present
    if {"ClaimNb","Exposure"}.issubset(df_test.columns):
        df_test["ClaimRate"] = df_test["ClaimNb"] / df_test["Exposure"]

# ============================================================
# 2) Feature engineering transformer (compact, parametric)
# ============================================================
class FeatureRules(BaseEstimator, TransformerMixin):
    def __init__(self, schema=SCHEMA):
        self.schema = schema
        self.area_map = {a: i+1 for i, a in enumerate(schema["ordered_area"])}

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        # Log-transform rules (map: column → transformation)
        if "Density" in X:
            X["Density_log1p"] = np.log1p(X["Density"])

        # Generic binning (loop through schema bins)
        binning_specs = {
            "BonusMalus": self.schema["bonus_bins"],
            "DrivAge": self.schema["driv_bins"],
            "VehAge": self.schema["vehage_bins"]
        }
        for col, (bins, labels) in binning_specs.items():
            if col in X:
                X[f"{col}_bin"] = pd.cut(X[col], bins=bins, labels=labels, include_lowest=True)

        # VehPower: numeric, rank, and quantile bins
        if "VehPower" in X:
            X["VehPower_rank"] = X["VehPower"].rank(method="average")
            X["VehPower_bin"] = pd.qcut(
                X["VehPower"], q=self.schema["vehpower_q"],
                labels=[f"q{i+1}" for i in range(self.schema["vehpower_q"])]
            )

        # Area: ordered ordinal
        if "Area" in X:
            X["Area_ord"] = X["Area"].map(self.area_map).fillna(np.median(list(self.area_map.values())))

        # VehGas: simple binary
        if "VehGas" in X:
            X["VehGas_bin"] = (X["VehGas"] == "Diesel").astype(int)

        return X


rules = FeatureRules()
df_train_feat = rules.transform(df_train)
df_test_feat  = rules.transform(df_test) if has_external_test else None

# ============================================================
# 3) Build parametric column sets (auto + minimal explicit)
# ============================================================
def exists(cols, df):
    return [c for c in cols if c in df.columns]

def derive_feature_lists(df_feat, schema):
    num_feats = exists(
        schema["raw_numeric_candidates"] + ["VehPower_rank", "Density_log1p", "Exposure", "Area_ord"],
        df_feat
    )
    cat_feats = exists(schema["categorical_candidates"], df_feat)
    return num_feats, cat_feats

num_base, cat_base = derive_feature_lists(df_train_feat, SCHEMA)


# ColumnTransformer
preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler())
        ]), num_base),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("enc", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False))
        ]), cat_base),
    ],
    remainder="drop"
)

# ============================================================
# 4) Fit preprocess on TRAIN only, transform both
# ============================================================
X_train_mat = preprocess.fit_transform(df_train_feat)
X_test_mat  = preprocess.transform(df_test_feat) if has_external_test else None

# Recover feature names post-encoding (for PCA/loadings)
def get_feature_names(ct: ColumnTransformer) -> list:
    names = []
    # numeric passthrough pipeline has no names change besides scaler
    names += num_base
    # cat: get ohe names
    ohe = ct.named_transformers_["cat"].named_steps["enc"]
    ohe_names = ohe.get_feature_names_out(cat_base).tolist()
    names += ohe_names
    return names

feature_names = get_feature_names(preprocess)

# ============================================================
# 5) PCA with cumulative explained variance + feature attribution
# ============================================================
pca = PCA().fit(X_train_mat)
cum = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(8,4))
plt.plot(np.arange(1, len(cum)+1), cum, marker="o")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA – Cumulative Explained Variance")
plt.grid(True)
plt.show()

# choose PCs up to threshold
K = int(np.searchsorted(cum, SCHEMA["pca_var_threshold"]) + 1)

# feature “contribution” score: sum over first K PCs of |loading| * variance_fraction
# (maps correctly to original columns)
var_frac = pca.explained_variance_ratio_[:K].reshape(-1, 1)        # (K,1)
load_abs = np.abs(pca.components_[:K, :])                          # (K, n_features)
feat_scores = (load_abs * var_frac).sum(axis=0)                    # (n_features,)
top_idx = np.argsort(feat_scores)[::-1]
top20 = [(feature_names[i], float(feat_scores[i])) for i in top_idx[:20]]
print(f"Top features by PCA (up to {SCHEMA['pca_var_threshold']*100:.0f}% var):")
for name, score in top20:
    print(f"  {name:35s}  score={score:.5f}")

# 3D clustering over first 3 PCs (better visibility)
X_train_pca = pca.transform(X_train_mat)
kmeans = KMeans(n_clusters=4, random_state=0, n_init=10)
labels = kmeans.fit_predict(X_train_pca[:, :3])

fig = plt.figure(figsize=(6,5))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X_train_pca[:,0], X_train_pca[:,1], X_train_pca[:,2], c=labels, alpha=0.5, s=10)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_zlabel("PC3")
ax.set_title("KMeans on first 3 PCs (3D)")
plt.show()

# ============================================================
# 6) Model registry + custom scoring (penalize non-zeros)
# ============================================================

def weighted_mae_nonzero(y_true, y_pred, alpha=SCHEMA["alpha_nonzero_penalty"]):
    # weight non-zero ground truth higher
    w = 1.0 + alpha * (y_true != 0).astype(float)
    return np.average(np.abs(y_true - y_pred), weights=w)

def make_scorer_nonzero(alpha=SCHEMA["alpha_nonzero_penalty"]):
    # scorer for GridSearchCV (returns negative for 'greater_is_better=False')
    def _score(est, X, y):
        y_hat = est.predict(X)
        return -weighted_mae_nonzero(y, y_hat, alpha=alpha)
    return _score

# base preprocess cloned inside pipelines to keep them independent per model
base_pre = preprocess

MODEL_REGISTRY = {
    "DecisionTree": Pipeline([
        ("pre", clone(base_pre)),
        ("model", DecisionTreeRegressor(class_weight='balanced', random_state=0))
    ]),
    "NeuralNet": Pipeline([
        ("pre", clone(base_pre)),
        ("model", MLPRegressor(hidden_layer_sizes=(64,32),
                               max_iter=750, random_state=0, verbose=True))
    ])
}
# optional third models (feel free to add/replace)
EXTRA_MODELS = {
    "GBR": Pipeline([
        ("pre", clone(base_pre)),
        ("model", GradientBoostingRegressor(random_state=0))
    ]),
    "ExtraTrees": Pipeline([
        ("pre", clone(base_pre)),
        ("model", ExtraTreesRegressor(n_estimators=400, random_state=0, n_jobs=-1, verbose=0))
    ])
}

# modest grids (expand later)
GRIDS = {
    "DecisionTree": {"model__max_depth": [4, 6, 8, 12]},
    "NeuralNet": {
        "model__hidden_layer_sizes": [(64,32), (128,64), (128,64,32)],
        "model__alpha": [1e-4, 1e-3],
        "model__learning_rate_init": [1e-3, 5e-4]
    },
    "GBR": {"model__max_depth": [2,3], "model__n_estimators": [200,400], "model__learning_rate":[0.05,0.1]},
    "ExtraTrees": {"model__max_depth": [None, 12], "model__min_samples_leaf":[1,3]}
}

# ============================================================
# 7) Train/Val/Test split + CV; evaluate BOTH targets
# ============================================================
def run_model_selection(X_df, y, label):
    X_tr, X_te, y_tr, y_te = train_test_split(X_df, y, test_size=0.2, random_state=42)
    results = []
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scorer = make_scorer_nonzero()

    for name, pipe in {**MODEL_REGISTRY, **EXTRA_MODELS}.items():
        grid = GRIDS.get(name, {})
        gs = GridSearchCV(pipe, grid, cv=kf, scoring=scorer, n_jobs=-1, verbose=2)
        gs.fit(X_tr, y_tr)

        best = gs.best_estimator_
        # train / val cv score
        best_cv = -gs.best_score_

        # test score (on held-out split)
        y_hat = best.predict(X_te)
        test_mae = mean_absolute_error(y_te, y_hat)
        test_wmae = weighted_mae_nonzero(y_te, y_hat)

        print(f"[{label}] {name}: best_params={gs.best_params_}  CV_wMAE={best_cv:.6f}  "
              f"Test_MAE={test_mae:.6f}  Test_wMAE={test_wmae:.6f}")

        results.append((name, best, gs.best_params_, best_cv, test_mae, test_wmae))

    # pick best by weighted MAE on the test split
    results.sort(key=lambda t: t[-1])
    return results

# Use parametric feature set (no hard-coded interior names)
FEATURES = num_base + cat_base
X_df = df_train_feat[FEATURES].copy()

# 7a) ClaimNb (zero-heavy)
res_nb = run_model_selection(X_df, df_train["ClaimNb"], label="ClaimNb")

# 7b) ClaimRate (frequency)
res_rate = run_model_selection(X_df, df_train["ClaimRate"], label="ClaimRate")

best_nb_name, best_nb_est, *_ = res_nb[0]
best_rate_name, best_rate_est, *_ = res_rate[0]

print(f"\nBest for ClaimNb:   {best_nb_name}")
print(f"Best for ClaimRate: {best_rate_name}")

# ============================================================
# 8) Final test-set evaluation (if external test is available)
# ============================================================
if has_external_test:
    X_test_df = df_test_feat[FEATURES].copy()

    # ClaimNb only if present in test
    if "ClaimNb" in df_test.columns:
        y_test_nb = df_test["ClaimNb"]
        yhat_nb = best_nb_est.predict(X_test_df)
        print(f"[External TEST] ClaimNb  MAE={mean_absolute_error(y_test_nb, yhat_nb):.6f}  "
              f"wMAE={weighted_mae_nonzero(y_test_nb, yhat_nb):.6f}")

    # ClaimRate only if computable on test
    if {"ClaimNb","Exposure"}.issubset(df_test.columns):
        y_test_rate = df_test["ClaimNb"] / df_test["Exposure"]
        yhat_rate = best_rate_est.predict(X_test_df)
        print(f"[External TEST] ClaimRate  MAE={mean_absolute_error(y_test_rate, yhat_rate):.6f}  "
              f"wMAE={weighted_mae_nonzero(y_test_rate, yhat_rate):.6f}")